In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Load cleaned data without altering row order
df = pd.read_csv("appointments_clean.csv")

# Select only the permitted predictors
feature_cols = [
    "age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
    "reminder_sent", "booking_channel", "visit_type"
]
X = df[feature_cols].copy()

# Map binary column and one-hot encode categoricals
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])

y = df["no_show"]

print(f"X shape: {X.shape}")
print("Columns:", list(X.columns))

# 80/20 split with fixed seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Baseline: always predicting "Attended" (0)
test_no_shows = y_test.sum()
baseline_acc = (y_test == 0).mean()

print(f"\nTrain size: {len(X_train)}, Test size: {len(X_test)}")
print(f"Test no-shows: {test_no_shows} / {len(y_test)}")
print(f"Baseline accuracy: {baseline_acc:.4f}")

X shape: (990, 11)
Columns: ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']

Train size: 792, Test size: 198
Test no-shows: 49 / 198
Baseline accuracy: 0.7525


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score, roc_auc_score
)

# Scale using training statistics only
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Fit Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)

# Evaluate on the test set
y_pred = lr.predict(X_test_s)
y_prob = lr.predict_proba(X_test_s)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Precision:", round(precision_score(y_test, y_pred), 4))
print("Recall:", round(recall_score(y_test, y_pred), 4))
print("F1:", round(f1_score(y_test, y_pred), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

# Print coefficients in the column order of X
coef_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": lr.coef_[0],
    "scaler_sd": scaler.scale_
})
print("\n--- Coefficients ---")
print(coef_table.to_string(index=False))

# Odds ratio for reminder_sent (scaled vs. actual 0 -> 1 step)
rem_idx = list(X.columns).index("reminder_sent")
rem_coef = lr.coef_[0][rem_idx]
rem_scale = scaler.scale_[rem_idx]

odds_scaled = np.exp(rem_coef)
odds_actual = np.exp(rem_coef / rem_scale)
pct_reduction = (1 - odds_actual) * 100

print(f"\nreminder_sent coef: {rem_coef:.4f}, SD: {rem_scale:.4f}")
print(f"exp(coef) [1-SD change]: {odds_scaled:.4f}")
print(f"exp(coef / scale) [No -> Yes change]: {odds_actual:.4f} ({pct_reduction:.1f}% lower odds)")

Accuracy: 0.7879
Confusion Matrix:
 [[139  10]
 [ 32  17]]
Precision: 0.6296
Recall: 0.3469
F1: 0.4474
ROC-AUC: 0.7889

--- Coefficients ---
                feature  coefficient  scaler_sd
                    age    -0.253880  16.655863
            distance_km     0.253448   6.051810
                fee_inr     0.452761 184.253540
              lead_days     0.614471   8.614206
         prior_no_shows     0.649612   0.968578
          reminder_sent    -0.580669   0.494975
    booking_channel_App    -0.046804   0.486016
  booking_channel_Phone     0.215881   0.481046
booking_channel_Walk-In    -0.186364   0.435178
visit_type_Consultation     0.015658   0.492792
   visit_type_Follow-up    -0.015658   0.492792

reminder_sent coef: -0.5807, SD: 0.4950
exp(coef) [1-SD change]: 0.5595
exp(coef / scale) [No -> Yes change]: 0.3094 (69.1% lower odds)


**Task 3.2 Interpretation:**
* **Scaled vs. unscaled odds ratio:** `np.exp(coef) = 0.5595` reflects the change in no-show odds for a one-standard-deviation shift (`SD = 0.4950`) in the standardized feature, whereas dividing by `scaler.scale_` first (`np.exp(coef / scale) = 0.3094`) rescales the effect to a full 1-unit change from `No` (0) to `Yes` (1).
* **Observational caveat:** In this model, holding the other features constant, sending a reminder is associated with **69.1% lower odds** of a no-show (odds ratio `0.3094`); because these booking records are observational rather than from a randomized experiment, this indicates an association rather than proof that reminders cause fewer no-shows.

In [3]:
from sklearn.tree import DecisionTreeClassifier

# 1. Full-depth Decision Tree (no depth limit)
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)

print(f"Full-depth Tree (depth={dt_full.get_depth()}):")
print(f"  Train Accuracy: {accuracy_score(y_train, dt_full.predict(X_train)):.4f}")
print(f"  Test Accuracy:  {accuracy_score(y_test, dt_full.predict(X_test)):.4f}")

# 2. Constrained Decision Tree (max_depth=3)
dt_3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_3.fit(X_train, y_train)

dt3_test_pred = dt_3.predict(X_test)
dt3_test_prob = dt_3.predict_proba(X_test)[:, 1]

print("\nDepth-3 Tree:")
print(f"  Train Accuracy: {accuracy_score(y_train, dt_3.predict(X_train)):.4f}")
print(f"  Test Accuracy:  {accuracy_score(y_test, dt3_test_pred):.4f}")

Full-depth Tree (depth=15):
  Train Accuracy: 1.0000
  Test Accuracy:  0.6717

Depth-3 Tree:
  Train Accuracy: 0.8220
  Test Accuracy:  0.7778


**Task 3.3 Interpretation:**
The unconstrained tree (`depth=15`) achieves `1.0000` training accuracy but drops to `0.6717` on the test set (below the `0.7525` majority-class baseline), demonstrating severe overfitting where the model memorised training noise, whereas limiting `max_depth=3` narrows the gap (`0.8220` train vs. `0.7778` test) and generalizes much better.

In [4]:
from sklearn.ensemble import RandomForestClassifier

# Fit Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_test_pred = rf.predict(X_test)
rf_test_prob = rf.predict_proba(X_test)[:, 1]

print("Random Forest:")
print(f"  Train Accuracy: {accuracy_score(y_train, rf.predict(X_train)):.4f}")
print(f"  Test Accuracy:  {accuracy_score(y_test, rf_test_pred):.4f}")
print(f"  Test Recall:    {recall_score(y_test, rf_test_pred):.4f}")
print(f"  Test ROC-AUC:   {roc_auc_score(y_test, rf_test_prob):.4f}")

# Top 3 feature importances
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\n--- Top 3 Feature Importances ---")
print(importances.head(3).round(4))

# Comparison table across the three models
comparison_df = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, y_prob), 4)
    },
    {
        "Model": "Decision Tree (max_depth=3)",
        "Accuracy": round(accuracy_score(y_test, dt3_test_pred), 4),
        "Recall": round(recall_score(y_test, dt3_test_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, dt3_test_prob), 4)
    },
    {
        "Model": "Random Forest (100 trees)",
        "Accuracy": round(accuracy_score(y_test, rf_test_pred), 4),
        "Recall": round(recall_score(y_test, rf_test_pred), 4),
        "ROC-AUC": round(roc_auc_score(y_test, rf_test_prob), 4)
    }
])

print("\n--- Model Comparison (Test Set - For Reporting Only) ---")
print(comparison_df.to_string(index=False))

Random Forest:
  Train Accuracy: 1.0000
  Test Accuracy:  0.7475
  Test Recall:    0.3061
  Test ROC-AUC:   0.7057

--- Top 3 Feature Importances ---
lead_days      0.2199
age            0.1968
distance_km    0.1947
dtype: float64

--- Model Comparison (Test Set - For Reporting Only) ---
                      Model  Accuracy  Recall  ROC-AUC
        Logistic Regression    0.7879  0.3469   0.7889
Decision Tree (max_depth=3)    0.7778  0.2245   0.6997
  Random Forest (100 trees)    0.7475  0.3061   0.7057


**Task 3.4 Comparison Note:**
Across the reporting metrics, **Logistic Regression** performs best because it achieves the highest test ROC-AUC (`0.7889`), highest accuracy (`0.7879`), and highest recall (`0.3469`) of the three models; however, this table is for reporting only, and the formal model selection will be made strictly using 5-fold cross-validation on the training set in Task 3.5.

In [5]:
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline

# (a) Tune Random Forest with GridSearchCV on the training set only
param_grid = {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc"
)
grid_search.fit(X_train, y_train)

print(f"(a) Tuned RF best_params_: {grid_search.best_params_}")
print(f"(a) Tuned RF best_score_ (5-fold CV ROC-AUC): {grid_search.best_score_:.4f}")

# (b) Compare all models using 5-fold CV ROC-AUC on the training set
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

cv_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_dt3 = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_rf = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_rf_tuned = grid_search.best_score_

print("\n(b) 5-Fold Cross-Validated Training ROC-AUC:")
print(f"  Logistic Regression (Pipeline): {cv_lr:.4f}")
print(f"  Decision Tree (max_depth=3):    {cv_dt3:.4f}")
print(f"  Random Forest (untuned):        {cv_rf:.4f}")
print(f"  Random Forest (tuned):          {cv_rf_tuned:.4f}")

# (c) Out-of-fold training probabilities for the winner (Logistic Regression)
oof_probs = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

print("\n(c) Out-of-Fold Training Threshold Comparison:")
for thresh in [0.5, 0.35]:
    oof_preds = (oof_probs >= thresh).astype(int)
    prec = precision_score(y_train, oof_preds)
    rec = recall_score(y_train, oof_preds)
    f1 = f1_score(y_train, oof_preds)
    flagged = oof_preds.sum()
    print(f"  Threshold {thresh:.2f} -> Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}, Flagged: {flagged}")

# (d) Fit winner on full training set and evaluate ONCE on test set at threshold 0.35
chosen_threshold = 0.35
lr_pipe.fit(X_train, y_train)
test_probs = lr_pipe.predict_proba(X_test)[:, 1]
test_flags = (test_probs >= chosen_threshold).astype(int)

print(f"\n(d) Final Test Set Evaluation (Threshold = {chosen_threshold}):")
print(f"  Precision: {precision_score(y_test, test_flags):.4f}")
print(f"  Recall:    {recall_score(y_test, test_flags):.4f}")
print(f"  Flagged:   {test_flags.sum()} / {len(y_test)}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, test_probs):.4f}")

# (e) Save test_risk_scores.csv
risk_scores_df = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].values,
    "no_show": y_test.values,
    "risk_prob": test_probs,
    "flag": test_flags
})
risk_scores_df.to_csv("test_risk_scores.csv", index=False)
print(f"\n(e) Saved test_risk_scores.csv with shape {risk_scores_df.shape} and {risk_scores_df['flag'].sum()} flagged rows.")

(a) Tuned RF best_params_: {'max_depth': 3, 'n_estimators': 200}
(a) Tuned RF best_score_ (5-fold CV ROC-AUC): 0.7673

(b) 5-Fold Cross-Validated Training ROC-AUC:
  Logistic Regression (Pipeline): 0.7737
  Decision Tree (max_depth=3):    0.6784
  Random Forest (untuned):        0.7466
  Random Forest (tuned):          0.7673

(c) Out-of-Fold Training Threshold Comparison:
  Threshold 0.50 -> Precision: 0.5753, Recall: 0.2485, F1: 0.3471, Flagged: 73
  Threshold 0.35 -> Precision: 0.5130, Recall: 0.4675, F1: 0.4892, Flagged: 154

(d) Final Test Set Evaluation (Threshold = 0.35):
  Precision: 0.5714
  Recall:    0.5714
  Flagged:   49 / 198
  ROC-AUC:   0.7889

(e) Saved test_risk_scores.csv with shape (198, 4) and 49 flagged rows.


**Task 3.5 Selection & Threshold Interpretation:**
* **Model & threshold choice:** **Logistic Regression** wins with the highest 5-fold cross-validated training ROC-AUC (`~0.77`), and threshold **`0.35`** is selected over `0.50` because it achieves a higher out-of-fold F1 score (`~0.49` vs. `~0.35`) and substantially higher recall (`~0.47` vs. `~0.25`)—which is critical because missing a high-risk patient leaves an expensive doctor slot empty, whereas an extra reminder phone call only costs a minute of front-desk time.
* **Real-world threshold setting:** In practice, a clinic would set its probability threshold by balancing how many follow-up phone calls the front desk staff can realistically make in an evening against the financial and clinical cost of an empty appointment slot.